### Domain-adaptive pretraining (DAPT)

Continue RoBERTa-large's MLM pretraining on each unlabeled pool, then fine-tune with the standard recipe. Run order: setup, MLM cell (~1h FOMC, ~2.5h global), idiom probe, fine-tune, frozen probe.


### Colab Setup

In [3]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Key Imports

In [4]:
import pandas as pd
import torch
from sklearn.metrics import f1_score

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS, IDIOMS
from data.apt_pools import build_pools
from data.loader_twd_labelled import load_splits
from models.dapt import dapt
from models.frozen_probe import probe
from models.plm_finetune import finetune
from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
SEEDS = SHAH_SEEDS
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Continued pretraining (one pass per pool)

In [5]:
# "fomc" is the TWD pool alone
# "global" concatenates it with the WCB pool at use,
_fomc, _global = build_pools()

POOLS = {"fomc": _fomc, "global": _global}

ARMS = {
    "dapt-fomc:roberta-large": ("roberta-large", "fomc", 1, "dapt-fomc"),
    "dapt-global:roberta-large": ("roberta-large", "global", 1, "dapt-global"),
    # parked -- consistency check across encoders:
    # "dapt-fomc:bert-large-uncased": ("bert-large-uncased", "fomc", 1, "dapt-fomc-bert-large"),
    # "dapt-global:bert-large-uncased": ("bert-large-uncased", "global", 1, "dapt-global-bert-large"),
    # parked -- run only if the flat result needs the undertraining objection killed:
    # "dapt-fomc-x4:roberta-large": ("roberta-large", "fomc", 4, "dapt-fomc-x4"),
}

for arm, (enc, pool, epochs, dirname) in ARMS.items():
    save_dir = str(RESULTS_DIR / "models" / dirname)
    if os.path.isdir(save_dir):
        print(f"{arm}: already adapted at {save_dir}, skipping")
        continue
    sentences = POOLS[pool]["sentence"].astype(str).tolist()
    print(f"{arm}: {len(sentences):,} sentences, {epochs} epoch(s)", flush=True)
    dapt(
        sentences,
        model_name=SHAH_PLM[enc]["model_name"],
        epochs=epochs,
        save_dir=save_dir,
        device=DEVICE,
        verbose=True,
    )

downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmpzu36ovbi
  meeting_minutes: 230 docs -> 47,340 sentences
  speech: 1026 docs -> 107,548 sentences
  press_conference: 63 docs -> 24,750 sentences


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 31.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

twd: 166894 | wcb: 297388
cross-source duplicates dropped from wcb: 2
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


...

5768/val-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  404kB            

5768/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

5768/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  400kB            

5768/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

labelled key set: 25561
  twd: 2206 contaminated rows removed
  wcb: 22660 contaminated rows removed
fomc pool: 164,688 | global pool: 439,414
dapt-fomc:roberta-large: already adapted at /content/drive/MyDrive/thesis/models/dapt-fomc, skipping
dapt-global:roberta-large: already adapted at /content/drive/MyDrive/thesis/models/dapt-global, skipping


### Masked-idiom Test (pre vs post adaptation)

In [6]:
# Gambacorta et al. 2024 Appendix 1

from transformers import pipeline


def probe_idioms(model_path, idioms):
    mlm = pipeline("fill-mask", model=model_path, device=0 if DEVICE == "cuda" else -1)
    mask = mlm.tokenizer.mask_token
    rows = []
    for phrase, gold in idioms:
        top5 = [
            r["token_str"].strip().lower()
            for r in mlm(phrase.replace("[MASK]", mask), top_k=5)
        ]
        rows.append(
            dict(
                phrase=phrase, gold=gold, hit=gold.lower() in top5, top5="|".join(top5)
            )
        )
    del mlm
    torch.cuda.empty_cache()
    return rows


models = {enc: SHAH_PLM[enc]["model_name"] for enc, *_ in ARMS.values()}
models.update(
    {arm: str(RESULTS_DIR / "models" / d) for arm, (_, _, _, d) in ARMS.items()}
)

records = []
for label, path in models.items():
    rows = probe_idioms(path, IDIOMS)
    records += [dict(model=label, **r) for r in rows]
    print(f"{label}: {sum(r['hit'] for r in rows)}/{len(rows)}", flush=True)

idf = pd.DataFrame(records)
idf.to_csv(RESULTS_DIR / "idioms.csv", index=False)
print("saved ->", RESULTS_DIR / "idioms.csv")


# gained / lost vs the arm's own vanilla encoder
def hitset(m):
    return set(idf[(idf.model == m) & idf.hit].phrase)


for arm, (enc, *_) in ARMS.items():
    print(f"{arm} gained:", sorted(hitset(arm) - hitset(enc)))
    print(f"{arm} lost:  ", sorted(hitset(enc) - hitset(arm)))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

roberta-large: 69/100


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

dapt-fomc:roberta-large: 82/100


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

dapt-global:roberta-large: 85/100
saved -> /content/drive/MyDrive/thesis/idioms.csv
dapt-fomc:roberta-large gained: ['Accommodative [MASK] policy', 'Asset [MASK] program', 'Bretton [MASK] system', 'Countercyclical [MASK] buffer', 'Effective [MASK] rate', 'Exchange [MASK] pass-through', 'Financial [MASK] board', 'Foreign [MASK] intervention', 'Interest [MASK] on excess reserves', 'Marginal [MASK] facility', 'Open [MASK] operations', 'Overnight [MASK] facility', 'Secured [MASK] financing rate']
dapt-fomc:roberta-large lost:   []
dapt-global:roberta-large gained: ['Accommodative [MASK] policy', 'Asset [MASK] program', 'Countercyclical [MASK] buffer', 'Effective [MASK] rate', 'Efficient [MASK] hypothesis', 'Exchange [MASK] pass-through', 'Financial [MASK] board', 'Foreign [MASK] intervention', 'Interest [MASK] on excess reserves', 'Macroprudential [MASK] measures', 'Marginal [MASK] facility', 'Nominal [MASK] rate', 'Open [MASK] operations', 'Overnight [MASK] facility', 'Secured [MASK] fina

### Fine-tune adapted encoders (3 seeds each)

In [7]:
# fine-tune each adapted encoder with its vanilla winner's config. rows land
for arm, (enc, _, _, dirname) in ARMS.items():
    cfg = SHAH_PLM[enc]
    model_dir = str(RESULTS_DIR / "models" / dirname)
    for seed in SEEDS:
        if already_done(OUT, force=FORCE, model=arm, corpus="twd", seed=seed):
            print(f"{arm} seed {seed}: already done, skipping")
            continue
        train, test = load_splits("benchmark", seed=seed)
        model, tok_, metrics = finetune(
            train,
            model_name=model_dir,
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=arm,
            corpus="twd",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{arm} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()

dapt-fomc:roberta-large seed 5768: already done, skipping
dapt-fomc:roberta-large seed 78516: already done, skipping
dapt-fomc:roberta-large seed 944601: already done, skipping
dapt-global:roberta-large seed 5768: already done, skipping
dapt-global:roberta-large seed 78516: already done, skipping
dapt-global:roberta-large seed 944601: already done, skipping


### Frozen probe on adapted encoders (3 seeds, no fine-tuning)

In [8]:
# frozen probe on every adapted encoder: representation-level gain without
for arm, (enc, _, _, dirname) in ARMS.items():
    MODEL = f"frozen-{arm}"
    model_dir = str(RESULTS_DIR / "models" / dirname)
    for seed in SEEDS:
        if already_done(OUT, force=FORCE, model=MODEL, corpus="twd", seed=seed):
            print(f"{MODEL} seed {seed}: already done, skipping")
            continue
        train, test = load_splits("benchmark", seed=seed)
        pred = probe(train, test, model_name=model_dir, device=DEVICE, seed=seed)
        true = test["label"].to_list()
        save_result(
            OUT,
            model=MODEL,
            corpus="twd",
            seed=seed,
            epochs="",
            weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
            macro_f1=round(f1_score(true, pred, average="macro"), 4),
        )
        print(f"{MODEL} seed {seed}: macro={f1_score(true, pred, average='macro'):.4f}")

frozen-dapt-fomc:roberta-large seed 5768: already done, skipping
frozen-dapt-fomc:roberta-large seed 78516: already done, skipping
frozen-dapt-fomc:roberta-large seed 944601: already done, skipping
frozen-dapt-global:roberta-large seed 5768: already done, skipping
frozen-dapt-global:roberta-large seed 78516: already done, skipping
frozen-dapt-global:roberta-large seed 944601: already done, skipping
